# BA Audio Analysis Notebook – v2

Pipeline-Übersicht:
1. Ziel und Forschungsbezug
2. Setup und Konfiguration
3. Transkription (Whisper / gpt-4o-mini-transcribe)
4. Prosodische Merkmale (Sprechtempo, Lautstärke, Pausen, Silben/Sek.)
5. Emotionserkennung (emotion2vec)
6. JSON-Versionen v1 / v2 / v3 erzeugen
7. GPT-Experiment: Antworten vergleichen
8. Nächste Schritte

## 1. Ziel des Notebooks

Dieses Notebook bildet die vollständige erste Pipeline ab:
- Gesprochenes Audio in Text umwandeln
- Prosodische Merkmale extrahieren (Sprechtempo, Lautstärke, Pausen, Silben/Sekunde)
- Emotion aus dem Audio erkennen (emotion2vec)
- Informationen in drei JSON-Versionen strukturieren
- GPT mit verschiedenen JSON-Versionen befragen und Antworten vergleichen

Das Ziel ist zu untersuchen, ob eine KI empathischer reagiert, wenn sie neben dem Transkript auch prosodische und emotionale Informationen erhält.

## 2. Bezug zu den Forschungsfragen

- **FF1**: Wie können Speech-to-Text und prosodische Merkmale automatisch extrahiert werden?
  - Transkription via `gpt-4o-mini-transcribe`
  - Lautstärke via `librosa` (RMS)
  - Sprechtempo via Wörter/Min. und Silben/Sek.
  - Pausen via Silence Ratio
  - Emotion via `emotion2vec`

- **FF2**: Wie können diese Informationen strukturiert an eine KI weitergegeben werden?
  - Direkt als JSON im User-Prompt

- **FF3**: Wie entstehen daraus verschiedene JSON-Versionen für spätere Experimente?
  - v1 = nur Transkript (Baseline)
  - v2 = Transkript + einfache Prosodie
  - v3 = alles inkl. Emotion

- **FF4**: Experiment – Reagiert die KI empathischer mit mehr Kontext?

## 3. Setup und Konfiguration

Bibliotheken installieren (einmalig ausführen):
```bash
pip install openai python-dotenv librosa modelscope funasr torch torchaudio
```

In [ ]:
import json
import os
import re
from pathlib import Path

import numpy as np
import librosa
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY fehlt. Bitte in der .env-Datei setzen.")

client = OpenAI(api_key=api_key)

# Hier den Pfad zur Audiodatei anpassen
audio_path  = Path("../audio/gestresst.wav")
output_dir  = Path("../audio")

print(f"Audio file: {audio_path}")
print("Setup erfolgreich")

## 4. Transkription

Die Audiodatei wird mit `gpt-4o-mini-transcribe` in Text umgewandelt.

In [ ]:
with open(audio_path, "rb") as audio_file:
    transcript = client.audio.transcriptions.create(
        model="gpt-4o-mini-transcribe",
        file=audio_file,
    )

text = transcript.text.strip()

print("Transkript:")
print(text)

## 5. Prosodische Analyse

Folgende Merkmale werden berechnet:
- Dauer der Aufnahme
- Wörter pro Minute & Kategorie
- Silben pro Sekunde
- Lautstärke (RMS) & Kategorie
- Silence Ratio (Anteil stiller Frames)

In [ ]:
# --- Dauer ---
y, sr = librosa.load(str(audio_path))
duration_seconds = librosa.get_duration(y=y, sr=sr)

print(f"Dauer: {round(duration_seconds, 2)} Sek.")

In [ ]:
# --- Sprechtempo: Wörter pro Minute ---
word_count        = len(text.split())
words_per_minute  = (word_count / duration_seconds) * 60

if words_per_minute < 110:
    speech_speed = "slow"
elif words_per_minute < 160:
    speech_speed = "normal"
else:
    speech_speed = "fast"

print(f"Wörter: {word_count}")
print(f"Wörter/Min.: {round(words_per_minute, 2)}")
print(f"Kategorie Sprechtempo: {speech_speed}")

In [ ]:
# --- Sprechtempo: Silben pro Sekunde ---
def count_syllables(text: str) -> int:
    text = text.lower()
    vowel_groups = re.findall(r'[aeiouäöüy]+', text)
    return max(1, len(vowel_groups))

syllable_count       = count_syllables(text)
syllables_per_second = syllable_count / duration_seconds

print(f"Geschätzte Silben: {syllable_count}")
print(f"Silben/Sek.: {round(syllables_per_second, 2)}")

In [ ]:
# --- Lautstärke (RMS) ---
rms      = librosa.feature.rms(y=y)
mean_rms = float(np.mean(rms))
max_rms  = float(np.max(rms))

if mean_rms < 0.02:
    volume_category = "leise"
elif mean_rms < 0.07:
    volume_category = "normal"
else:
    volume_category = "laut"

print(f"RMS Mittelwert: {round(mean_rms, 4)}")
print(f"RMS Maximum:    {round(max_rms, 4)}")
print(f"Kategorie Lautstärke: {volume_category}")

In [ ]:
# --- Pausen / Silence Ratio ---
silence_threshold = 0.01
rms_frames        = rms[0]
silent_frames     = np.sum(rms_frames < silence_threshold)
silence_ratio     = float(silent_frames / len(rms_frames))

print(f"Stille-Anteil: {round(silence_ratio * 100, 1)} %")

## 6. Emotionserkennung mit emotion2vec

Voraussetzung: `pip install modelscope funasr torch torchaudio`

Verfügbare Emotionslabels (Basis-Modell): neutral, happy, sad, angry, fearful, disgusted, surprised

In [ ]:
from modelscope.pipelines import pipeline
from modelscope.utils.constant import Tasks

emotion_pipeline = pipeline(
    task=Tasks.emotion_recognition,
    model="iic/emotion2vec_base_finetuned",
)

emotion_result = emotion_pipeline(str(audio_path), output_dir="./emotion_output")

scores    = emotion_result[0]["scores"]
labels    = emotion_result[0]["labels"]
top_idx   = int(np.argmax(scores))

emotion_label = labels[top_idx]
emotion_score = round(float(scores[top_idx]), 4)

print(f"Erkannte Emotion: {emotion_label}  (Score: {emotion_score})")
print("\nAlle Scores:")
for label, score in zip(labels, scores):
    print(f"  {label}: {round(float(score), 4)}")

## 7. JSON-Versionen v1 / v2 / v3

- **v1**: Nur Transkript → Baseline (KI sieht nur den Text)
- **v2**: Transkript + einfache Prosodie
- **v3**: Alles inkl. Emotion und detaillierte Prosodie

In [ ]:
# v1 – nur Transkript (Baseline)
json_v1 = {
    "version": "v1",
    "transcript": text
}

# v2 – Transkript + einfache Prosodie
json_v2 = {
    "version": "v2",
    "transcript": text,
    "prosody": {
        "speech_rate_wpm":      round(words_per_minute, 2),
        "speech_rate_category": speech_speed,
        "silence_ratio_pct":    round(silence_ratio * 100, 1),
        "volume_category":      volume_category
    }
}

# v3 – alles inkl. Emotion
json_v3 = {
    "version": "v3",
    "transcript": text,
    "prosody": {
        "duration_seconds":      round(duration_seconds, 2),
        "word_count":            word_count,
        "syllables_per_second":  round(syllables_per_second, 2),
        "speech_rate_wpm":       round(words_per_minute, 2),
        "speech_rate_category":  speech_speed,
        "silence_ratio_pct":     round(silence_ratio * 100, 1),
        "volume_rms_mean":       round(mean_rms, 4),
        "volume_category":       volume_category
    },
    "emotion": {
        "label": emotion_label,
        "score": emotion_score
    }
}

for v in [json_v1, json_v2, json_v3]:
    print(json.dumps(v, indent=2, ensure_ascii=False))
    print("---")

# Speichern
stem = audio_path.stem
for version, payload in [("v1", json_v1), ("v2", json_v2), ("v3", json_v3)]:
    out = output_dir / f"{stem}_{version}.json"
    with open(out, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)
    print(f"Gespeichert: {out}")

## 8. GPT-Experiment: Antworten über v1 / v2 / v3 vergleichen

Hier wird GPT mit jeder JSON-Version befragt. Die Antworten werden verglichen, um zu sehen, ob mehr Kontext zu empathischeren Reaktionen führt.

**Temperatur**: Niedrig (z.B. 0.3) = konsistenter, Hoch (z.B. 1.0) = kreativer/variabler

In [ ]:
# Temperatur hier anpassen für Experimente
TEMPERATURE = 0.3

system_prompt = (
    "Du bist ein einfühlsamer Assistent. "
    "Antworte auf die Aussage der Person kurz und empathisch auf Deutsch. "
    "Wenn du prosodische oder emotionale Informationen erhältst, berücksichtige diese in deiner Antwort."
)

def ask_gpt(payload: dict, temperature: float = TEMPERATURE) -> str:
    user_content = json.dumps(payload, ensure_ascii=False)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=temperature,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_content}
        ]
    )
    return response.choices[0].message.content.strip()


results_comparison = {}

for version, payload in [("v1", json_v1), ("v2", json_v2), ("v3", json_v3)]:
    answer = ask_gpt(payload)
    results_comparison[version] = answer
    print(f"\n{'='*40}")
    print(f"Antwort {version} (Temperatur={TEMPERATURE}):")
    print(f"{'='*40}")
    print(answer)

# Vergleich speichern
comparison_path = output_dir / f"{stem}_gpt_comparison.json"
with open(comparison_path, "w", encoding="utf-8") as f:
    json.dump({
        "audio_file":  audio_path.name,
        "temperature": TEMPERATURE,
        "answers":     results_comparison
    }, f, indent=2, ensure_ascii=False)

print(f"\nVergleich gespeichert: {comparison_path}")

## 9. Nächste Schritte

- [ ] Mehrere Audiodateien (verschiedene Emotionen) durch die Pipeline laufen lassen
- [ ] Verschiedene Temperaturen systematisch testen (z.B. 0.0, 0.3, 0.7, 1.0)
- [ ] Tonfall via Pitch-Analyse ergänzen (`librosa.yin` oder `librosa.piptrack`)
- [ ] Experiment für FF4 definieren und Bewertungsschema für Empathie entwickeln
- [ ] Ergebnisse in Tabelle zusammenfassen (z.B. mit `pandas`)